<a href="https://colab.research.google.com/github/LikhithR916/BIS_Lab/blob/main/Parallel%20Cellular%20Algorithm/1BM24CS147_IndoorAirQualityVentilationOptimization_ParallelCellularOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
from concurrent.futures import ThreadPoolExecutor

# Parameters
ROWS, COLS = 3, 3
MAX_ITER = 100
MIN_VENT, MAX_VENT = 20, 100

np.random.seed(42)

# Fitness weights: Air quality has highest priority
CO2_WEIGHT = 0.90
TEMP_WEIGHT = 0.04
HUMIDITY_WEIGHT = 0.04
ENERGY_WEIGHT = 0.02

OUTDOOR_CO2 = 420
CO2_LIMIT = 1000

# Initial room conditions
temperature = np.array([
    [25, 28, 24],
    [30, 26, 27],
    [23, 29, 25]
], dtype=float)

humidity = np.array([
    [45, 60, 40],
    [65, 50, 55],
    [42, 62, 48]
], dtype=float)

co2 = np.array([
    [800, 1200, 650],
    [1500, 900, 1300],
    [700, 1400, 850]
], dtype=float)

# Initialize ventilation
ventilation = np.random.uniform(
    MIN_VENT, MAX_VENT, (ROWS, COLS)
)

initial_grid = ventilation.copy()


# Find Moore neighbourhood
def get_neighbors(grid, r, c):

    neighbors = []

    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:

            nr, nc = r + dr, c + dc

            if (dr != 0 or dc != 0) and \
               0 <= nr < ROWS and 0 <= nc < COLS:

                neighbors.append(grid[nr, nc])

    return neighbors


# Estimate CO2 after ventilation
def estimate_co2(r, c, vent, grid):

    neighbors = get_neighbors(grid, r, c)

    avg = np.mean(neighbors) if neighbors else vent

    effective_vent = 0.8 * vent + 0.2 * avg

    return OUTDOOR_CO2 + (
        co2[r, c] - OUTDOOR_CO2
    ) * np.exp(-0.025 * effective_vent)


# Fitness function
def fitness(r, c, vent, grid):

    final_co2 = estimate_co2(r, c, vent, grid)

    neighbors = get_neighbors(grid, r, c)

    avg = np.mean(neighbors) if neighbors else vent

    effective_vent = 0.8 * vent + 0.2 * avg

    final_temp = temperature[r, c] - 0.015 * effective_vent

    final_humidity = humidity[r, c] + (
        45 - humidity[r, c]
    ) * effective_vent / 100 * 0.15

    # CO2 penalty: lower CO2 gives lower penalty
    co2_penalty = (
        (final_co2 - OUTDOOR_CO2) /
        (CO2_LIMIT - OUTDOOR_CO2)
    ) ** 2

    temp_penalty = ((final_temp - 24) / 10) ** 2

    humidity_penalty = ((final_humidity - 50) / 50) ** 2

    energy = (vent / 100) ** 2

    return (
        CO2_WEIGHT * co2_penalty
        + TEMP_WEIGHT * temp_penalty
        + HUMIDITY_WEIGHT * humidity_penalty
        + ENERGY_WEIGHT * energy
    )


# Update individual cell
def update_cell(args):

    r, c, old_grid = args

    current = old_grid[r, c]

    neighbors = get_neighbors(old_grid, r, c)

    avg = np.mean(neighbors) if neighbors else current

    # Never reduce ventilation below its initial level
    minimum = initial_grid[r, c]

    candidates = [
        current,
        current + 10,
        current + 20,
        current + 30,
        avg,
        100
    ]

    candidates = np.clip(
        candidates, minimum, MAX_VENT
    )

    best_vent = current
    best_score = fitness(r, c, current, old_grid)

    for candidate in candidates:

        score = fitness(r, c, candidate, old_grid)

        if score < best_score:
            best_score = score
            best_vent = candidate

    return r, c, best_vent


# Parallel Cellular Algorithm
def parallel_cellular_algorithm(grid):

    with ThreadPoolExecutor(max_workers=ROWS * COLS) as executor:

        for iteration in range(MAX_ITER):

            old_grid = grid.copy()

            tasks = [
                (r, c, old_grid)
                for r in range(ROWS)
                for c in range(COLS)
            ]

            # Parallel cell updates
            results = list(
                executor.map(update_cell, tasks)
            )

            new_grid = old_grid.copy()

            for r, c, value in results:
                new_grid[r, c] = value

            grid = new_grid

    return grid


# Calculate CO2 grid
def calculate_co2_grid(grid):

    result = np.zeros((ROWS, COLS))

    for r in range(ROWS):
        for c in range(COLS):

            result[r, c] = estimate_co2(
                r, c, grid[r, c], grid
            )

    return result


# Calculate total fitness
def total_fitness(grid):

    total = 0

    for r in range(ROWS):
        for c in range(COLS):

            total += fitness(r, c, grid[r, c], grid)

    return total


# Run optimization
optimized_grid = parallel_cellular_algorithm(
    ventilation.copy()
)

# Calculate initial and final CO2
initial_co2 = calculate_co2_grid(initial_grid)
final_co2 = calculate_co2_grid(optimized_grid)

# Calculate energy consumption
initial_energy = np.sum((initial_grid / 100) ** 2)
final_energy = np.sum((optimized_grid / 100) ** 2)

# Calculate fitness
initial_fitness = total_fitness(initial_grid)
final_fitness = total_fitness(optimized_grid)

# Display results
print("\nINDOOR AIR QUALITY VENTILATION OPTIMIZATION")
print("=" * 50)

print("\nInitial Ventilation Levels (%):")
print(np.round(initial_grid, 2))

print("\nOptimized Ventilation Levels (%):")
print(np.round(optimized_grid, 2))

print("\nAverage CO2 Concentration:")
print("Initial:", round(np.mean(initial_co2), 2), "ppm")
print("Final:  ", round(np.mean(final_co2), 2), "ppm")

print("\nEnergy Consumption Index:")
print("Initial:", round(initial_energy, 4))
print("Final:  ", round(final_energy, 4))

print("\nFitness Value:")
print("Initial:", round(initial_fitness, 5))
print("Final:  ", round(final_fitness, 5))

print("\nOptimization completed successfully!")



INDOOR AIR QUALITY VENTILATION OPTIMIZATION

Initial Ventilation Levels (%):
[[49.96 96.06 78.56]
 [67.89 32.48 32.48]
 [24.65 89.29 68.09]]

Optimized Ventilation Levels (%):
[[ 79.96 100.    78.56]
 [100.    88.73 100.  ]
 [ 63.22 100.    88.09]]

Average CO2 Concentration:
Initial: 561.72 ppm
Final:   480.78 ppm

Energy Consumption Index:
Initial: 3.7831
Final:   7.2195

Fitness Value:
Initial: 0.74268
Final:   0.27016

Optimization completed successfully!
